Updating/editing work done by Debby Irtania

## **Pre-Processing**


---



### **Import Packages**

In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import rapidfuzz
from rapidfuzz import process, fuzz
import os

### **Connect to Google Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## **Load in Data**


---



### **Charity Commission**
Load charity commission data

In [ ]:
# load in charity classification
char_class = pd.read_csv('/content/drive/MyDrive/DISSERTATION/DataFiles/CharityCommission/charity_classification.txt', sep = "\t")

# load charity area
char_area = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/CharityCommission/charity_area_operation.txt",sep="\t")

# load in charity detail
char_detail = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/CharityCommission/charity.txt", sep="\t", on_bad_lines="skip", low_memory=False)

# load charity annual report
char_ann = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/CharityCommission/char_income.txt",sep="\t",on_bad_lines="skip", low_memory=False)

In [ ]:
char_detail.columns.tolist()

['date_of_extract',
 'organisation_number',
 'registered_charity_number',
 'linked_charity_number',
 'charity_name',
 'charity_type',
 'charity_registration_status',
 'date_of_registration',
 'date_of_removal',
 'charity_reporting_status',
 'latest_acc_fin_period_start_date',
 'latest_acc_fin_period_end_date',
 'latest_income',
 'latest_expenditure',
 'charity_contact_address1',
 'charity_contact_address2',
 'charity_contact_address3',
 'charity_contact_address4',
 'charity_contact_address5',
 'charity_contact_postcode',
 'charity_contact_phone',
 'charity_contact_email',
 'charity_contact_web',
 'charity_company_registration_number',
 'charity_insolvent',
 'charity_in_administration',
 'charity_previously_excepted',
 'charity_is_cdf_or_cif',
 'charity_is_cio',
 'cio_is_dissolved',
 'date_cio_dissolution_notice',
 'charity_activities',
 'charity_gift_aid',
 'charity_has_land']

### **Company House**
Load company house data

In [ ]:
import shutil
shutil.copy(
    "/content/drive/MyDrive/DISSERTATION/DataFiles/CompanyHouse/companies_house.csv",
    "/content/companies_house.csv"
)

cols = ["CompanyName", " CompanyNumber", "RegAddress.PostCode",
        "CompanyCategory", "CompanyStatus", "CountryOfOrigin",
        "IncorporationDate", "DissolutionDate",
        "SICCode.SicText_1", "SICCode.SicText_2",
        "SICCode.SicText_3", "SICCode.SicText_4"]

df_coh = pd.read_csv("/content/companies_house.csv",
                     usecols=cols,
                     on_bad_lines="skip",
                     low_memory=False)

df_coh = df_coh.rename(columns={' CompanyNumber': 'company_number'})
print(df_coh.shape)

# parse dates
df_coh["IncorporationDate"] = pd.to_datetime(df_coh["IncorporationDate"], errors="coerce")
df_coh["DissolutionDate"] = pd.to_datetime(df_coh["DissolutionDate"], errors="coerce")

(5698175, 12)


### **ONSPD**
load ONSPD data

In [ ]:
# load ONSPD data
onspd = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv")

/tmp/ipykernel_6815/331898789.py:2: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  onspd = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv")


### **Local Authority**
Load in local authority data

In [ ]:
# load la data
la = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/LocalAuthorities/LocalAuthorityNames.csv")

## **Filtering Functions**

---



### **Charity Commission Filtering**

In [ ]:
class CharityCommissionFilters:
    def __init__(self, char_class, char_detail, char_area):
        self.char_class = char_class
        self.char_detail = char_detail
        self.char_area = char_area

    # filters classification data to identify which charities work with young people only (strict) or young people among other beneficiaries (broad)
    # also filters based on what the charity does and how it helps its beneficiaries
    def filter_class_youth(char_class,who_value="Children/young People",mode='strict',keep_what=None,keep_how=None):
        # filter rows to 'Who' classification_type
        df_who = char_class[char_class['classification_type'] == 'Who']
        # unique who descriptiosn per org
        desc_per_org = df_who.groupby('organisation_number')['classification_description'].unique().reset_index()
        # strict filtering
        if mode == 'strict':
            # keep only orgs that soley list 'Children/young People' beneficiaries
            only_youth = desc_per_org[(desc_per_org['classification_description'].apply(len) == 1) & (desc_per_org['classification_description'].apply(lambda x: x[0] == who_value))]
        # broad filtering
        elif mode == 'broad':
            # keep orgs with 'Children/young People' among other beneficiaries
            only_youth = desc_per_org[desc_per_org['classification_description'].apply(lambda x: who_value in x)]
        else:
            # raise error input other than strict or broad
            raise ValueError("mode must be 'strict' or 'broad'")
        # list of organization numbers that qualified from the 'Who' filter
        youth_orgs = only_youth['organisation_number']
        # filter organizatiosn to only those whose org number is in youth_orgs
        df_who_final = df_who[df_who['organisation_number'].isin(youth_orgs)].copy()
        # extract 'What' and 'How' classifications and rename for clarity
        df_what = char_class[char_class["classification_type"]=="What"].rename(columns={'classification_description': 'classification_what'})
        df_how = char_class[char_class["classification_type"]=="How"].rename(columns={'classification_description': 'classification_how'})
        # merge the youth orgs with the 'How' classifications
        df_class1 = pd.merge(df_who_final,df_how[['organisation_number', 'classification_how']],on='organisation_number',how='left')
        # merge the youth orfs with the 'What' classifications
        df_class2 = pd.merge(
            df_class1,
            df_what[['organisation_number', 'classification_what']],
            on='organisation_number',
            how='left'
        )
        # apply optional filters (can apply one or both)
        if keep_how is not None and keep_what is not None:
            df_class_filtered = df_class2[
                (df_class2['classification_how'].isin(keep_how)) &
                (df_class2['classification_what'].isin(keep_what))
            ]
        elif keep_how is not None:
            df_class_filtered = df_class2[df_class2['classification_how'].isin(keep_how)]
        elif keep_what is not None:
            df_class_filtered = df_class2[df_class2['classification_what'].isin(keep_what)]
        else:
            df_class_filtered = df_class2
        # make dummy columns, its basically one-hot encoding for How & What
        df_how_dummy = pd.crosstab(df_class_filtered['organisation_number'], df_class_filtered['classification_how'])
        df_how_dummy = df_how_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
        df_what_dummy = pd.crosstab(df_class_filtered['organisation_number'], df_class_filtered['classification_what'])
        df_what_dummy = df_what_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
        # combine the dummy tables
        df_class_final = df_how_dummy.merge(df_what_dummy, on='organisation_number', how='outer')
        # attach charity number and original Who classification
        # if broad
        if mode == "broad":
            # keep just one row per org, remove duplicates
            df_who_final_unique = df_who_final.drop_duplicates("organisation_number")
        # if strict
        else:
            df_who_final_unique = df_who_final
        df_class_final1 = df_class_final.merge(
            df_who_final_unique[['organisation_number', 'registered_charity_number', 'classification_description']],
            on='organisation_number',
            how='left'
        )
        # reorder columns
        select_cols = (
            ['organisation_number', 'registered_charity_number', 'classification_description'] +
            [col for col in df_class_final.columns if col != 'organisation_number']
        )
        df_class_clean = df_class_final1[select_cols]
        return df_class_clean

    #  merges the filtered charity data with the full charity detail dataset and returns only registered charities
    def merge_charity_details(df_class_clean, char_detail, start_date=None, end_date=None):
        # filter to relevant columns
        cols = [
            'organisation_number', 'charity_name', 'charity_type',
            'charity_registration_status', 'charity_contact_postcode',
            "charity_company_registration_number", "charity_is_cio",
            "latest_income", "latest_expenditure",
            "date_of_registration", "date_of_removal"
        ]
        # merge classification data with register info
        df_detail = pd.merge(
            df_class_clean,
            char_detail[cols],
            on='organisation_number',
            how='left'
        )
        # add company numbers for CIOs from external file
        cio = pd.read_csv("/content/drive/MyDrive/DISSERTATION/DataFiles/CompanyHouse/cio_company_numbers.csv")
        cio = cio[cio['company_number'].astype(str).str.isnumeric()]
        cio['charity_number'] = cio['company_number'].astype(int)
        df_detail = df_detail.merge(
            cio[['charity_number', 'company_number']],
            left_on='registered_charity_number',
            right_on='charity_number',
            how='left',
            suffixes=('', '_lookup'))
        # fill missing company registration numbers (for CIOs)
        df_detail['charity_company_registration_number'] = df_detail['charity_company_registration_number'].fillna(df_detail['company_number'])
        # clean up unnecessary columns after merge
        df_detail = df_detail.drop(columns=['charity_number', 'company_number'], errors='ignore')

        # transform to datetime type
        df_detail['date_of_registration'] = pd.to_datetime(df_detail['date_of_registration'], errors='coerce')
        df_detail['date_of_removal'] = pd.to_datetime(df_detail['date_of_removal'], errors='coerce')
        # check that charities are within start and end date
        if start_date is not None and end_date is not None:
            df_detail_clean = df_detail[
                (df_detail['date_of_registration'] <= end_date) &
                (df_detail['date_of_removal'].isna() | (df_detail['date_of_removal'] >= start_date))
            ].copy()
        else:
            # fallback to original behaviour
            df_detail_clean = df_detail[df_detail['charity_registration_status'] == "Registered"].copy()

        df_detail_clean = df_detail_clean.drop_duplicates(subset='organisation_number', keep='first')
        # return df
        return df_detail_clean

    #  merges the cleaned charity dataset with geographic area information from the Charity Commission
    # standardizes postcodes for mapping
    def merge_area_of_operation(df_detail_clean, char_area):
        # merge with area level data from the Charity Commission on org number
        df_area = pd.merge(
            df_detail_clean,
            char_area[['organisation_number', 'geographic_area_type', 'geographic_area_description',
                      'parent_geographic_area_description', 'welsh_ind']],
            on='organisation_number',
            how='left'
        )
        # filter to only charities operating in England, exluding those operating at the Welsh or UK county level
        df_area = df_area[(df_area["welsh_ind"] == False) &(df_area["geographic_area_type"] != "Country")]
        # remove rows with missing postcode
        df_area1 = df_area.dropna(subset=["charity_contact_postcode"]).copy()
        # standardize postcodes - removing whitespace and converting to uppercase
        df_area1["charity_contact_postcode"] = (df_area1["charity_contact_postcode"].astype(str).str.strip().str.upper())
        # remove postcodes that don’t contain any digits
        df_area2 = df_area1[df_area1['charity_contact_postcode'].apply(lambda x: any(char.isdigit() for char in x))]
        # clean and fix postcode errors (Debby manually validated)
        def clean_postcode(pc):
            # manual corrections
            pc = pc.replace('WA6 2EH', 'WA6 8EJ')   # 1st Newto & Kingsley charities – checked with website
            pc = pc.replace('CA16 3SW', 'CA16 6QU') # Appleby Guide Hut – cross-checked with website
            pc = pc.replace('SG4 4PB', 'SG5 4PB')   # 1ST STOTFOLD SCOUT GROUP
            pc = pc.replace('EX14 2QF', 'EX14 1QF') # BUSY BEES
            pc = pc.replace('ME16 6TL', 'ME15 6TL') # Little Fawns
            pc = pc.replace('SY11 3SW', 'SY11 3JS') # 1ST GOBOWEN SCOUT GROUP – from Charity Commission
            pc = pc.replace('SE16 6EA', 'SE16 5EA') # PLAYSHACK PLAYGROUP
            pc = pc.replace('RM7 0AH', 'RM3 0BP')   # TAKE A KNIFE SAVE A LIFE
            pc = pc.replace('SG6 6DD', 'SG6 4DD')   # M7 EDUCATION – NEED CROSSCHECK
            pc = pc.replace('WD7 7QL', 'WD7 7LQ')   # Charity No. 1195380
            pc = pc.replace('CH61 9UG', 'CH61 7UG') # Charity No. 1201178
            # further general cleanup
            pc = pc.strip().upper().replace(' ', '')
            # fixing common errors (letter O vs number 0)
            pc = pc.replace('CRO', 'CR0')  # fix CR0 typo
            pc = pc.replace('P02', 'PO2')  # fix PO2 typo
            # return formatted postcode only if it’s a valid length
            if len(pc) < 5 or len(pc) > 7:
                return 'INVALID'
            else:
                # add space before last three characters for standarization
                return pc[:-3] + ' ' + pc[-3:]
        # clean each postcode
        df_area2["charity_contact_postcode"] = df_area2["charity_contact_postcode"].apply(clean_postcode)
        # filter only keeping valid UK postcode format
        postcode_pattern = r'^[A-Z]{1,2}\d{1,2}[A-Z]?\s\d[A-Z]{2}$'
        df_area_clean = df_area2[df_area2['charity_contact_postcode'].str.match(postcode_pattern)]
        # return cleaned df
        return df_area_clean

    # links each charity's postcode to its MSOA and Local Authority (LAD) using ONS postcode lookups
    def cross_reference_deduplicate(df_area3,lookup_path,lookup_name_path):
        # function to clean postcode in lookup data from ONS postcode to LA
        def clean_postcode(pc):
            pc = str(pc).strip().upper().replace(' ', '')
            if len(pc) < 5 or len(pc) > 7:
                return 'INVALID'
            else:
                return pc[:-3] + ' ' + pc[-3:]
        # load data lookup ONS Aug
        lookup = pd.read_csv(lookup_path)
        lookup_LA = lookup[['pcd2', 'oslaua', 'msoa21']].drop_duplicates()
        lookup_LA['pcd2'] = lookup_LA['pcd2'].astype(str).apply(clean_postcode)
        # merge postcode from charity dataset
        df_area_lookup = df_area3.merge(
            lookup_LA[['pcd2', 'oslaua', 'msoa21']],
            left_on='charity_contact_postcode',
            right_on='pcd2',
            how='left'
        )
        # merge with LA name
        lookup_name = pd.read_csv(lookup_name_path)
        df_area_lookup1 = df_area_lookup.merge(
            lookup_name[['LAD23CD', 'LAD23NM']],
            left_on='oslaua',
            right_on='LAD23CD',
            how='left'
        )
        # keep only England MSOAs, start with "E"
        df_area_lookup1 = df_area_lookup1[df_area_lookup1["msoa21"].fillna("").str.startswith("E")]
        # fuzzy match geographic_area_description (from charity) with LAD name
        def cross_reference(row):
            geo_desc = str(row['geographic_area_description']).lower().strip()
            lad_name = str(row['LAD23NM']).lower().strip()
            if geo_desc == lad_name:
                return "MATCH", 100  # exact match
            elif str(row['geographic_area_type']).lower().strip() == 'region': #if the area type "region", we assume the charity postcode is the area of the operation of the charity
                return "MATCH", 100
            else:
                # calculate fuzzy matching score (Debbby: average of 3 methods for more robust result)
                score = (fuzz.partial_ratio(geo_desc, lad_name) + fuzz.token_sort_ratio(geo_desc, lad_name) + fuzz.token_set_ratio(geo_desc, lad_name)) / 3
                if score >= 80:
                    return "MATCH", score
                else:
                    return "UNMATCH", score
        # apply fuzzy matching to all rows
        df_area_lookup1[['cross_reference', 'fuzzy_score']] = df_area_lookup1.apply(cross_reference, axis=1, result_type='expand')
        # remove duplicates from each org + MSOA, keep one row (MATCH prioritized)
        df_filter = (
            df_area_lookup1
            .assign(priority=lambda d: d['cross_reference'].eq('MATCH').astype(int))
            .sort_values(['organisation_number', 'msoa21', 'priority'], ascending=[True, True, False]) #false here for sorting by "match"
            .drop_duplicates(subset=['organisation_number', 'msoa21'])
        )
        # return filtered df
        return df_filter

    # adds new columns for independent schools, early years providers, and income size category
    def flag_ind_early_income(df,ind_school_path,income_col="latest_income",ind_school_threshold=90,early_years_threshold=85,early_years_list=None):
        # import process and fuzz
        from rapidfuzz import process, fuzz
        # load list of independent schools, used for fuzzy match
        indp_school = pd.read_csv(ind_school_path, sep=",", encoding="latin1")
        # cleanse school name
        schools_list = (indp_school['EstablishmentName'].dropna().str.lower().str.strip().unique().tolist())
        # function of fuzzy matching for independent schools
        def is_independent_school(charity_name, choices, threshold=ind_school_threshold):
            # cleanse charity name
            name = str(charity_name).lower().strip()
            # if the name already contains "independent"
            if "independent" in name:
                return "Independent School", 100
            # fuzzy match score with reference list
            match, score, _ = process.extractOne(name, choices, scorer=fuzz.token_set_ratio)
            # Exact / strong match
            if name == match.lower().strip():
                return "Independent School", 100
            elif score >= threshold:
                return "Independent School", score
            else:
                return "Other", score
        # list of early years keywords
        if early_years_list is None:
            early_years = ['pre-school', 'preschool', 'nursery', 'early years', 'playgroup','toddler group', 'toddler school', 'kindergarten', 'childcare','day nursery', 'day care']
        else:
            early_years = early_years_list
        # function for fuzzy matcher for early years providers
        def flag_early_years_fuzzy(name, threshold=early_years_threshold):
            name_lower = str(name).lower().strip()
            match, score, _ = process.extractOne(name_lower, early_years, scorer=fuzz.token_set_ratio)
            if score >= threshold:
                return "Early Years", score
            else:
                return "Other", score
        # apply both fuzzy matchers to the charity name
        df = df.copy()
        df[['indp_school_flag', 'fuzzy_score_indp']] = df['charity_name'].apply(lambda x: pd.Series(is_independent_school(x, schools_list)))
        df[['early_years_flag', 'fuzzy_score_early_years']] = df['charity_name'].apply(lambda x: pd.Series(flag_early_years_fuzzy(x)))
        # add income category column from income thresholds
        if income_col in df.columns:
            df[income_col] = df[income_col].fillna(0)
            def categorize_income(income):
                if income < 10_000:
                    return "Micro"
                elif income < 100_000:
                    return "Small"
                elif income < 1_000_000:
                    return "Medium"
                else:
                    return "Large"
            df["income_category"] = df[income_col].apply(categorize_income)
        # if income not defined raise error
        else:
            print(f"WARNING: {income_col} income is not defined.")
        # return df
        return df

    # applies final filtering based on location match (cross-reference), independent school status, early years provider (?), income level
    def filter_charities(df,drop_unmatch=True,drop_ind_school=True,drop_early_years=True,drop_micro=True):
        filtered = df.copy()
        # keep only charities with valid postcode area match
        if drop_unmatch and "cross_reference" in filtered.columns:
            filtered = filtered[filtered["cross_reference"] == "MATCH"]
        # remove independent schools (if has flag)
        if drop_ind_school and "indp_school_flag" in filtered.columns:
            filtered = filtered[filtered["indp_school_flag"] != "Independent School"]
        # remove early years providers (if has flag)
        if drop_early_years and "early_years_flag" in filtered.columns:
            filtered = filtered[filtered["early_years_flag"] != "Early Years"]
        # remove micro-income charities (if income information)
        if drop_micro and "income_category" in filtered.columns:
            filtered = filtered[filtered["income_category"] != "Micro"]
        # return df
        return filtered

### **Company House Filters**

In [ ]:
class CompanyHouseFilters:
    def __init__(self, df):
        self.df = self.load_companies_house(df)

    # column standardization
    def load_companies_house(self, df_coh):
        # rename columns
        df = df_coh.rename(columns={
            ' CompanyNumber': 'company_number',
            'RegAddress.PostCode': 'postcode',
            'RegAddress.Country': 'Country',
            'CompanyCategory': 'company_category',
            'CompanyStatus': 'company_status',
            'CountryOfOrigin': 'country_origin',
            'SICCode.SicText_1': 'sic_text_1',
            'SICCode.SicText_2': 'sic_text_2',
            'SICCode.SicText_3': 'sic_text_3',
            'SICCode.SicText_4': 'sic_text_4'
        })
        # return df
        return df

    # filters the Companies House dataset to identify organizations likely to be involved in youth provision based on: company legal structure, company status, industry classification
    def filter_companies_by_sic_legal(self, df=None, valid_sic_list_core=None, mode="strict", start_date=None, end_date=None):
        if df is None:
          df = self.df
        # list of youth related SIC
        if valid_sic_list_core is None:
            valid_sic_list_core = [
            # Section P - Education
            '85510', # Sports and recreation education
            '85520', # Cultural education
            '85590', # Other education n.e.c.
            '85600', # Educational support services
            # Section R - Arts, entertainment, recreation
            '90010', # Performing arts
            '90020', # Support activities to performing arts
            '90030', # Artistic creation
            '90040', # Operation of arts facilities
            '91011', # Library activities
            '93110', # Operation of sports facilities
            '93120', # Activities of sport clubs
            '93199', # Other sports activities
            '93290', # Other amusement and recreation activities n.e.c.
            # Section S - Other service activities
            '94990'  # Activities of other membership organisations n.e.c.
            ]
        # keep only companies with acceptable legal form
        category_keep = ['PRI/LTD BY GUAR/NSC (Private, limited by guarantee, no share capital)','Community Interest Company',"PRI/LBG/NSC (Private, Limited by guarantee, no share capital, use of 'Limited' exemption)"]
        df_filtered = df[df['company_category'].isin(category_keep)].copy()
        # date-based active filter (overlap with the given window), replaces status=='Active' check
        if start_date is not None and end_date is not None:
            df_filtered['IncorporationDate'] = pd.to_datetime(df_filtered['IncorporationDate'], errors='coerce')
            df_filtered['DissolutionDate'] = pd.to_datetime(df_filtered['DissolutionDate'], errors='coerce')
            df_filtered = df_filtered[
                (df_filtered['IncorporationDate'] <= end_date) &
                (df_filtered['DissolutionDate'].isna() | (df_filtered['DissolutionDate'] >= start_date))
            ]
        else:
            # fallback to original behaviour
            df_filtered = df_filtered[df_filtered['company_status'] == 'Active']
        if mode == "broad":
            return df_filtered
        # if in strict assumption mode
        # extract sic code
        for i in range(1, 5):
            sic_text_col = f'sic_text_{i}'
            sic_code_col = f'sic_code_{i}'
            if sic_text_col in df_filtered.columns:
                df_filtered[sic_code_col] = df_filtered[sic_text_col].str.split(' - ').str[0]
        sic_cols = [f'sic_code_{i}' for i in range(1,5) if f'sic_code_{i}' in df_filtered.columns]
        # filter with only valid SIC code
        df_filtered['has_only_valid_sic'] = df_filtered[sic_cols].apply(lambda row: all(str(sic) in valid_sic_list_core for sic in row if pd.notnull(sic)),axis=1)
        # return df
        return df_filtered[df_filtered['has_only_valid_sic']].copy()

    # cleans company postcodes, then merges them with MSOA and Local Authority codes using ONS postcode
    def lookup_postcode(self, df, lookup_path, lookup_name_path):
        # clean the postcode format
        def clean_postcode(pc):
            if pd.isnull(pc):
                return pc
            pc = str(pc).strip().upper().replace(' ', '')
            if len(pc) < 5 or len(pc) > 7:
                return pc
            return pc[:-3] + ' ' + pc[-3:]
        # apply postcode cleaning
        df['postcode_clean'] = df['postcode'].apply(clean_postcode)
        postcode_pattern = r'^[A-Z]{1,2}[0-9][0-9A-Z]?\s?[0-9][A-Z]{2}$'
        df['valid_pc'] = df['postcode_clean'].str.match(postcode_pattern)
        df_valid = df[df['valid_pc'] & df['postcode_clean'].notna()].copy()
        # load postcode-to-area lookup, from ONS
        lookup = pd.read_csv(lookup_path)
        lookup_LA = lookup[['pcd2', 'oslaua', 'msoa21']].drop_duplicates()
        lookup_LA['pcd2'] = lookup_LA['pcd2'].astype(str).apply(clean_postcode)
        # merge with postcode lookup
        df_merge = df_valid.merge(
            lookup_LA,
            left_on='postcode_clean',
            right_on='pcd2',
            how='left'
        )
        # keep only England MSOAs
        df_merge = df_merge.dropna(subset=['msoa21'])
        # starts with an "E"
        df_merge = df_merge[df_merge['msoa21'].str.startswith('E')].copy()
        # merge with local authority names
        lookup_name = pd.read_csv(lookup_name_path)
        df_merge = df_merge.merge(
            lookup_name[['LAD23CD', 'LAD23NM']],
            left_on='oslaua',
            right_on='LAD23CD',
            how='left'
        )
        # return df
        return df_merge

    # removes companies that are also charities
    def remove_charity_duplicates(self, df_companies, df_charity_numbers):
        # list unique list of charity numbers
        charity_numbers = df_charity_numbers['charity_company_registration_number'].astype(str).unique()
        # is the company also in charity num?
        df_companies['match_flag'] = df_companies['company_number'].astype(str).isin(charity_numbers)
        # return df
        return df_companies[~df_companies['match_flag']].copy()

    # adds flags for independent schools, early years providers
    def flag_formal_early(self, df, school_list_path, org_name_col='CompanyName', school_threshold=90, early_years_threshold=85, early_years_list=None):
        # load list of formal schools (fuzzy match)
        school_df = pd.read_csv(school_list_path, sep=",", encoding="latin1", low_memory=False)
        school_names = (school_df['EstablishmentName'].dropna().str.lower().str.strip().unique().tolist())
        # function of fuzzy matching for formal schools
        def is_formal_school(name, choices, threshold=school_threshold):
            name = str(name).lower().strip()
            match, score, _ = process.extractOne(name, choices, scorer=fuzz.token_set_ratio)
            if name == match.lower().strip():
                return "Formal School", 100
            elif score >= threshold:
                return "Formal School", score
            else:
                return "Other", score
        # list of early years keywords
        if early_years_list is None:
            early_years = ['pre-school', 'preschool', 'nursery', 'early years', 'playgroup','toddler group', 'toddler school', 'kindergarten', 'childcare','day nursery', 'day care']
        else:
            early_years = early_years_list
        # function for fuzzy matcher for early years providers
        def flag_early_years_fuzzy(name, threshold=early_years_threshold):
            name_lower = str(name).lower().strip()
            match, score, _ = process.extractOne(name_lower, early_years, scorer=fuzz.token_set_ratio)
            if score >= threshold:
                return "Early Years", score
            else:
                return "Other", score
        # apply both fuzzy matchers to the organization name
        df = df.copy()
        df[org_name_col] = df[org_name_col].astype(str).str.lower().str.strip()
        df[['formal_school_flag', 'fuzzy_score_school']] = df[org_name_col].apply(lambda x: pd.Series(is_formal_school(x, school_names)))
        df[['early_years_flag', 'fuzzy_score_early_years']] = df[org_name_col].apply(lambda x: pd.Series(flag_early_years_fuzzy(x)))
        # add exclude_school_flag column for filtering if needed
        df['exclude_school_flag'] = df.apply(lambda row: "Yes" if (row['formal_school_flag'] == "Formal School" or row['early_years_flag'] == "Early Years") else "No", axis=1)
        # return df
        return df

    # processes and filters 360Giving grants data
    # standardizes column names, converts award dates, applies optional filters and extracts recipient/funder IDs for future merging
    def filter_grants(self, df_grant, min_amount=None, start_date=None, end_date=None):
        df_grants = pd.read_csv(df_grant)
        df = df_grants.rename(columns={
            "Title": "title",
            "Description": "description",
            "Amount Awarded": "amount_awarded",
            "Award Date": "award_date",
            "Recipient Org:Identifier": "recipient_identifier",
            "Recipient Org:Name": "recipient_name",
            "Recipient Org:Charity Number": "recipient_charity_number",
            "Recipient Org:Company Number": "recipient_company_number",
            "Recipient Org: Org Type (additional data)": "recipient_org_type",
            "Recipient Org: Canonical Org ID (additional data)": "recipient_canonical_id",
            "Funding Org:Identifier": "funder_identifier",
            "Funding Org:Name": "funder_name",
            "Funding Org:Charity Number": "funder_charity_number",
            "Funding Org:Company Number": "funder_company_number",
            "Funding Org: Org Type (additional data)": "funder_org_type",
            "Funding Org: Canonical Org ID (additional data)": "funder_canonical_id",
            "License": "license",
            "Note See http://grantnav.threesixtygiving.org/datasets/ for further license information.": "license_note"
        })
        df["award_date"] = pd.to_datetime(df["award_date"], errors='coerce')

        if min_amount is not None:
            df = df[df["amount_awarded"] >= min_amount]
        if start_date is not None:
            df = df[df["award_date"] >= start_date]
        if end_date is not None:
            df = df[df["award_date"] <= end_date]

        id_parts_recipient = df["recipient_identifier"].str.extract(r"GB-(\w+)-([A-Za-z0-9]+)")
        id_parts_funder = df["funder_identifier"].str.extract(r"GB-(\w+)-([A-Za-z0-9]+)")
        df["recipient_source"] = id_parts_recipient[0].map({"CHC": "Charity Commission", "COH": "Companies House"})
        df["recipient_number"] = id_parts_recipient[1]
        df["funder_number"] = id_parts_funder[1]
        return df

    # cross-checks company records with the 360Giving grants dataset, checks if a company has ever received or funded a grant
    def cross_reference_360giving(self, df_comp, df_grant):
        # get recipient company numbers from grants
        recipient_company_numbers = (df_grant['recipient_number'].dropna().astype(str).str.strip().unique())
        # get funder company numbers
        funder_company_numbers = (df_grant['funder_number'].dropna().astype(str).str.strip().unique())
        # flag if company matched with grants data from 360 giving as recipient
        df_comp['match_360giving'] = df_comp['company_number'].astype(str).isin(recipient_company_numbers)
        # flag if company matched with grants data from 360 giving as funder
        df_comp['match_360giving_fund'] = df_comp['company_number'].astype(str).isin(funder_company_numbers)
        # return df
        return df_comp

## **Data Wrangling**


---



In [ ]:
years = {
    "20162017": ['2016-09-01', '2017-07-31'],
    "20182019": ['2018-09-01', '2019-07-31'],
    "20212022": ['2021-09-01', '2022-07-31'],
    "20222023": ['2022-09-01', '2023-07-31'],
    "20242025": ['2024-09-01', '2025-07-31'],
}

### **Charity Commission Strict Assumptions**

In [ ]:
years = {
    "20162017": ['2016-09-01', '2017-07-31'],
    "20182019": ['2018-09-01', '2019-07-31'],
    "20212022": ['2021-09-01', '2022-07-31'],
    "20222023": ['2022-09-01', '2023-07-31'],
    "20242025": ['2024-09-01', '2025-07-31'],
}

#INCLUSION
# define "how" and "what"
keep_how = ['Provides Buildings/facilities/open Space', 'Provides Services']
keep_what = ['Education/training', 'Arts/culture/heritage/science', 'Amateur Sport', 'Recreation']

df_class_clean = CharityCommissionFilters.filter_class_youth(
    char_class,
    who_value="Children/young People",
    mode="strict",
    keep_what=keep_what,
    keep_how=keep_how
)

results = {}

for period_label, (start, end) in years.items():
    startdate = pd.Timestamp(start)
    enddate = pd.Timestamp(end)

    # apply function of charity details, using this period's window
    df_detail_clean = CharityCommissionFilters.merge_charity_details(df_class_clean, char_detail, startdate, enddate)

    # EXLCUSION
    # apply function of area details, includes filtered england only
    df_area_clean = CharityCommissionFilters.merge_area_of_operation(df_detail_clean, char_area)

    # apply location cross reference
    df_filter = CharityCommissionFilters.cross_reference_deduplicate(
        df_area_clean,
        lookup_path="/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv",
        lookup_name_path="/content/drive/MyDrive/DISSERTATION/DataFiles/LocalAuthorities/LocalAuthorityNames.csv"
    )

    # apply flagging function for independent school/early years/income
    df_flagged = CharityCommissionFilters.flag_ind_early_income(
        df_filter,
        ind_school_path="/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentSchools/independent_school.csv"
    )
    # FINAL
    # final filtering
    df_final_chc_strict = CharityCommissionFilters.filter_charities(
        df_flagged, drop_unmatch=True, drop_early_years=True, drop_ind_school=True, drop_micro=True
    )

    results[period_label] = df_final_chc_strict

    # save with period label in the path/filename
    df_final_chc_strict.to_csv(f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/CHCStrict/df_final_chc_strict{period_label}.csv", index=False)

/tmp/ipykernel_6815/1630004561.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_how_dummy = df_how_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
/tmp/ipykernel_6815/1630004561.py:57: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_what_dummy = df_what_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
/tmp/ipykernel_6815/1630004561.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_area2["charity_contact_postcode"] = df_area2["charity_contact_postcode"].apply(clean_postcode)
/tmp/ipykernel_6815/1630004561.py:192: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.r

### **Charity Commission Broad Assumptions**

In [ ]:
years = {
    "20162017": ['2016-09-01', '2017-07-31'],
    "20182019": ['2018-09-01', '2019-07-31'],
    "20212022": ['2021-09-01', '2022-07-31'],
    "20222023": ['2022-09-01', '2023-07-31'],
    "20242025": ['2024-09-01', '2025-07-31'],
}

# INCLUSION
# define "how" and "what"
keep_how = ['Provides Buildings/facilities/open Space', 'Provides Services']
keep_what = ['Education/training','Arts/culture/heritage/science','Amateur Sport','Recreation']
# use function filter class `broad` mode
df_class_clean_b = CharityCommissionFilters.filter_class_youth(char_class,who_value="Children/young People",mode="broad",keep_what=keep_what,keep_how=keep_how)

results = {}

for period_label, (start, end) in years.items():
    startdate = pd.Timestamp(start)
    enddate = pd.Timestamp(end)

    # apply function of charity details, and filter only registered charity
    df_detail_clean_b = CharityCommissionFilters.merge_charity_details(df_class_clean_b, char_detail, startdate, enddate)
    #EXLCUSION
    # apply function of area details, this include filtered england only
    df_area_clean_b = CharityCommissionFilters.merge_area_of_operation(df_detail_clean_b, char_area)
    # apply cross reference of location
    df_filter_b = CharityCommissionFilters.cross_reference_deduplicate(df_area_clean_b,lookup_path="/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv",lookup_name_path="/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/LA_UA.csv")
    # apply flagging function for independent school/early years/income
    df_flagged_b = CharityCommissionFilters.flag_ind_early_income(df_filter_b,ind_school_path="/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentSchools/independent_school.csv")
    # FINAL FILTER
    # apply function for final filtering of charity dataset (it is optional for the parameter that want to be used)
    df_final_chc_broad = CharityCommissionFilters.filter_charities(df_flagged_b,drop_unmatch=True,drop_early_years=True,drop_ind_school=True,drop_micro=True)

    results[period_label] = df_final_chc_broad

    # save with period label in the path/filename
    df_final_chc_broad.to_csv(f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/CHCBroad/df_final_chc_broad{period_label}.csv", index=False)

/tmp/ipykernel_6815/1630004561.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_how_dummy = df_how_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
/tmp/ipykernel_6815/1630004561.py:57: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_what_dummy = df_what_dummy.applymap(lambda x: 1 if x > 0 else 0).reset_index()
/tmp/ipykernel_6815/1630004561.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_area2["charity_contact_postcode"] = df_area2["charity_contact_postcode"].apply(clean_postcode)
/tmp/ipykernel_6815/1630004561.py:192: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.r

### **Companies House Strict Assumptions**

In [ ]:
chf = CompanyHouseFilters(df_coh)

results_coh = {}

for period_label, (start, end) in years.items():
    startdate = pd.Timestamp(start)
    enddate = pd.Timestamp(end)

    df_coh_filt = chf.filter_companies_by_sic_legal(start_date=startdate, end_date=enddate)

    # load grant data for this period's window
    df_grant_filt_x = chf.filter_grants(
        "/content/drive/MyDrive/DISSERTATION/DataFiles/CompanyHouse/grants.csv",
        min_amount=0, start_date=startdate, end_date=enddate
    )

    df_coh_cross_x = chf.cross_reference_360giving(df_coh_filt, df_grant_filt_x)
    df_coh_inc = df_coh_cross_x[df_coh_cross_x['match_360giving']]

    df_coh_area_x = chf.lookup_postcode(
        df_coh_inc,
        "/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv",
        "/content/drive/MyDrive/DISSERTATION/DataFiles/LocalAuthorities/LocalAuthorityNames.csv"
    )

    charity_path = f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/CHCBroad/df_final_chc_broad{period_label}.csv"
    df_final_chc_broad_period = pd.read_csv(charity_path)

    df_coh_dup_x = chf.remove_charity_duplicates(df_coh_area_x, df_final_chc_broad_period)

    df_coh_flag_x = chf.flag_formal_early(
        df_coh_dup_x, school_list_path="/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentSchools/independent_school.csv"
    )

    df_coh_final_strict = df_coh_flag_x[df_coh_flag_x["exclude_school_flag"] == "No"].copy()

    results_coh[period_label] = df_coh_final_strict
    df_coh_final_strict.to_csv(f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/COHStrict/df_coh_final_strict{period_label}.csv", index=False)

/tmp/ipykernel_6815/754680977.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['postcode_clean'] = df['postcode'].apply(clean_postcode)
/tmp/ipykernel_6815/754680977.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valid_pc'] = df['postcode_clean'].str.match(postcode_pattern)
/tmp/ipykernel_6815/754680977.py:93: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.read_csv(lookup_path)
/tmp/ipykernel_6815/754680977.py:8

### **Companies House Broad Assumptions**

In [ ]:
#INCLUSION
chf = CompanyHouseFilters(df_coh)
df_coh = chf.load_companies_house(df_coh)

results_coh_broad = {}

for period_label, (start, end) in years.items():
    startdate = pd.Timestamp(start)
    enddate = pd.Timestamp(end)

    # load grant data for this period's window
    df_grant_filt_x = chf.filter_grants(
        "/content/drive/MyDrive/DISSERTATION/DataFiles/CompanyHouse/grants.csv",
        min_amount=0, start_date=startdate, end_date=enddate
    )

    # cross reference the company number from companies dataset to grant data
    df_cross_b = chf.cross_reference_360giving(df_coh, df_grant_filt_x)
    # filter only match company
    df_coh_filt_b = df_cross_b[df_cross_b['match_360giving']]
    # filter legal type
    df_coh_legal_b = chf.filter_companies_by_sic_legal(df=df_coh_filt_b, mode="broad", start_date=startdate, end_date=enddate)

    # EXCLUSION

    # apply function of area details -> include lookup to LAD/MSOA
    df_coh_area_b = chf.lookup_postcode(df_coh_legal_b, "/content/drive/MyDrive/DISSERTATION/DataFiles/ONSPD/ONSPDAugust2025.csv", "/content/drive/MyDrive/DISSERTATION/DataFiles/LocalAuthorities/LocalAuthorityNames.csv")
    print(f"After postcode: {len(df_coh_area_b)}")
    # remove duplicate if the org already registered as charity
    charity_path = f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/CHCBroad/df_final_chc_broad{period_label}.csv"
    df_final_chc_broad_period = pd.read_csv(charity_path)
    df_coh_dup_b = chf.remove_charity_duplicates(df_coh_area_b, df_final_chc_broad_period)
    print(f"After charity dedup: {len(df_coh_dup_b)}")
    # flag formal school or early years school
    df_coh_flag_b = chf.flag_formal_early(df_coh_dup_b, school_list_path="/content/drive/MyDrive/DISSERTATION/DataFiles/IndependentSchools/independent_school.csv")
    print(f"After school flag: {len(df_coh_flag_b)}")

    # FILTER FILTERS

    # remove formal school or early years school
    df_coh_final_broad = df_coh_flag_b[df_coh_flag_b["exclude_school_flag"]=="No"].copy()
    print(f"Broad final: {len(df_coh_final_broad)}")

    results_coh_broad[period_label] = df_coh_final_broad
    df_coh_final_broad.to_csv(f"/content/drive/MyDrive/DISSERTATION/DataFiles/YouthProvisionLists/COHBroad/df_coh_final_broad{period_label}.csv", index=False)

/tmp/ipykernel_6815/754680977.py:93: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.read_csv(lookup_path)


After postcode: 114
After charity dedup: 73
After school flag: 73
Broad final: 71


/tmp/ipykernel_6815/754680977.py:93: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.read_csv(lookup_path)


After postcode: 161
After charity dedup: 114


/tmp/ipykernel_6815/761123165.py:31: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final_chc_broad_period = pd.read_csv(charity_path)


...

After school flag: 267
Broad final: 262


/tmp/ipykernel_6815/754680977.py:93: DtypeWarning: Columns (18,31,39,44,52) have mixed types. Specify dtype option on import or set low_memory=False.
  lookup = pd.read_csv(lookup_path)


After postcode: 281
After charity dedup: 253


/tmp/ipykernel_6815/761123165.py:31: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final_chc_broad_period = pd.read_csv(charity_path)


After school flag: 253
Broad final: 250
